<a href="https://colab.research.google.com/github/AMJAMAITHILI/DL_LAB/blob/main/internal2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

10. Implement RNN for predicting the next character
11. Implement BERT model  
12. Implement GAN model on MNIST.

In [1]:
from keras.models import Sequential
from keras.layers import SimpleRNN, Dense
import numpy as np

# Define vocabulary
vocab = ['d', 'e', 'p', '<stop>']
char2idx = {c: i for i, c in enumerate(vocab)}
idx2char = {i: c for c, i in char2idx.items()}

# Example input sequence: "dep"
sequence = ['d', 'e', 'p']
target = ['e', 'p', '<stop>']  # Next characters

# One-hot encode input and target
X = np.eye(len(vocab))[[char2idx[c] for c in sequence]].reshape(len(sequence), 1, len(vocab))
y = np.eye(len(vocab))[[char2idx[c] for c in target]]
# Build RNN model
model = Sequential([
    SimpleRNN(8, input_shape=(1, len(vocab))),
    Dense(len(vocab), activation='softmax')  # softmax output
])

model.compile(loss='categorical_crossentropy', optimizer='adam')  # cross-entropy loss

# Train the model
model.fit(X, y, epochs=200, verbose=0)

# Predict next character after 'p'
test = np.eye(len(vocab))[char2idx['p']].reshape(1, 1, len(vocab))
pred = model.predict(test)
print("Next char prediction:", idx2char[np.argmax(pred)])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
Next char prediction: <stop>


In [7]:
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf

# Step 1: Add from_pt=True to handle the PyTorch weights
model_name = 'textattack/bert-base-uncased-SST-2'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = TFBertForSequenceClassification.from_pretrained(model_name, from_pt=True)

# The rest of your code remains the same
text = "I hate my life!"
inputs = tokenizer(text, return_tensors='tf', truncation=True, padding=True, max_length=512)

outputs = model(inputs)
probabilities = tf.nn.softmax(outputs.logits, axis=-1)
predicted_class = tf.argmax(probabilities, axis=-1).numpy()[0]

print(f"Predicted Sentiment: {'Positive' if predicted_class == 1 else 'Negative'}")

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

All the weights of TFBertForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


Predicted Sentiment: Negative


In [4]:
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LeakyReLU, Reshape, Flatten
from tensorflow.keras.optimizers import Adam

# Load and normalize MNIST data
(X_train, _), _ = mnist.load_data()
X_train = (X_train - 127.5) / 127.5  # Scale to [-1, 1]
X_train = X_train.reshape(-1, 784)

# Optimizer
opt = Adam(0.0002, 0.5)

# --- Generator function ---
def build_generator():
    model = Sequential([
        Dense(128, input_dim=100),
        LeakyReLU(0.2),
        Dense(784, activation='tanh'),  # Output shape: 28x28 = 784
    ])
    return model

# --- Discriminator function ---
def build_discriminator():
    model = Sequential([
        Dense(128, input_shape=(784,)),
        LeakyReLU(0.2),
        Dense(1, activation='sigmoid')  # Binary output: real (1) or fake (0)
    ])
    model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build models
generator = build_generator()
discriminator = build_discriminator()

# --- Train GAN ---
noise = np.random.normal(0, 1, (128, 100))          # Random noise
fake_images = generator.predict(noise)              # Generate fake images

real_images = X_train[np.random.randint(0, X_train.shape[0], 128)]  # Sample real images

# Labels for training
real_labels = np.ones((128, 1))
fake_labels = np.zeros((128, 1))

# Train discriminator
d_loss_real = discriminator.train_on_batch(real_images, real_labels)
d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

# Print discriminator performance
print("Discriminator accuracy on real:", d_loss_real[1]*100)
print("Discriminator accuracy on fake:", d_loss_fake[1]*100)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Discriminator accuracy on real: 5.46875
Discriminator accuracy on fake: 30.078125
